In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone

import config.config_binance as config
from backtest.backtester import run_backtest
from data_loader import fetch_historical_data, get_binance_client

pd.set_option('future.no_silent_downcasting', True)


In [ ]:
# Data loader
start_dt = datetime(2024, 7, 13, tzinfo=timezone.utc)
start_ms = int(start_dt.timestamp() * 1000)

end_dt   = datetime.now(tz=timezone.utc)
end_ms   = int(end_dt.timestamp() * 1000)
df = fetch_historical_data(
            symbol=config.TRADING_SYMBOL, interval="1m", start_str=start_ms, end_str=end_ms
        )
df = df.set_index('open_time')
df = df.resample('1min').ffill()
df

In [ ]:
lag = 120        # minutes
df_lagged = df.copy()
df_lagged.index = df_lagged.index + pd.Timedelta(minutes=lag)   # shift timestamps forward

In [ ]:
def shift_signals(sig, bars=1):
    """Trade at the *next* bar after a signal."""
    return sig.shift(bars).fillna(0).astype(bool).astype(bool)

def simple_backtest(price, entries, exits):
    """Thin wrapper around vectorbt to keep code minimal."""
    import vectorbt as vbt
    pf = vbt.Portfolio.from_signals(price, entries, exits,
                                    init_cash=1_000,                       # €
                                    fees=0.0005)                           # 5 bp
    return pf

In [ ]:
# Sample strategy - 1
fast, slow = 50, 200
df_lagged['sma_fast'] = df_lagged.close.astype(float).rolling(fast).mean()
df_lagged['sma_slow'] = df_lagged.close.astype(float).rolling(slow).mean()

longs_raw  = df_lagged.sma_fast > df_lagged.sma_slow
shorts_raw = df_lagged.sma_fast < df_lagged.sma_slow

# shift by one bar if you want “trade next candle”
entries = shift_signals(longs_raw)
exits   = shift_signals(shorts_raw)

price = df_lagged['close'].astype('float64')
assert price.index.equals(entries.index)          # same timeline
assert price.index.tz is None                     # tz-naïve ✓
assert entries.dtype == bool and exits.dtype == bool
assert price.dtype == 'float64'

pf_sma  = simple_backtest(df_lagged.close, entries, exits)
print(pf_sma)

In [ ]:
stats = pf_sma.stats()
check_keys = ['Total Return [%]', 'Sharpe Ratio', 'Max Drawdown [%]', 'Sortino Ratio', 'Calmar Ratio']
print(stats[check_keys])
# duration = (pf_sma.wrapper.end_dt - pf_sma.wrapper.start_dt).days
# print(f"Back-test length: {duration} days")
idx_start = pf_sma.wrapper.index[0]
idx_end   = pf_sma.wrapper.index[-1]
duration  = idx_end - idx_start          # Timedelta
print("Back-test length:", duration)

In [ ]:
# full trade blotter in tabular form
pf_sma.trades.records_readable

# interactive equity curve & underwater plot
pf_sma.plot().show()

# iterate over the index if you need date stamps
for ts in pf_sma.wrapper.index[:5]:
    print(ts)

In [ ]:
# Simple lin. comb. prices
df['median'] = (df['high'] + df['low']) / 2
df['typical_price'] = (df['high'] + df['low'] + df['close']) / 3
df['weighted_close'] = (df['high'] + df['low'] + 2 * df['close']) / 4
df['ohlc_ave'] = (df['open'] + df['high'] + df['low'] + df['close']) / 4
df['high-low_range'] = df['high'] - df['low']
df['body_size'] = df['close'] - df['open']
df['upper_wick'] = df['high'] - np.maximum(df['open'], df['close'])
df['lower_wick'] = np.minimum(df['open'], df['close']) - df['low']

In [ ]:
# Return & momentum features
df["ret_1"] = df["close"].pct_change()
df["log_ret_1"] = np.log(df["close"]).diff()
for w in (5, 15, 30, 60):                                  # rolling windows
    df[f"ret_{w}"] = df["close"].pct_change(w)
    df[f"mom_{w}"] = df["close"].diff(w)                 # momentum
    df[f"roc_{w}"] = df["close"].pct_change(w) * 100     # rate of change %

In [ ]:
# Moving-average based signals
for w in (10, 20, 50, 200):
    df[f"sma_{w}"] = df["close"].rolling(w).mean()
    df[f"ema_{w}"] = df["close"].ewm(span=w, adjust=False).mean()
    df[f"close_above_sma_{w}"] = (df["close"] > df[f"sma_{w}"]).astype(int)
    df[f"distance_sma_{w}"] = df["close"] / df[f"sma_{w}"] - 1

In [ ]:
# Volatility metrics
df["hl_range"] = df["high"] - df["low"]
df["atr_14"] = df["hl_range"].rolling(14).mean()
df["rolling_std_30"] = df["ret_1"].rolling(30).std()
df["parkinson_vol_30"] = (
    (np.log(df["high"]/df["low"])**2).rolling(30).mean() * (1/(4*np.log(2)))
).pow(0.5)          # high-low-only estimator

In [ ]:
# Volume and Liquidity features
df["volume_rank_30"] = df["volume"].rank(
    pct=True, 
    method="max"
    ).rolling(30).apply(lambda x: x.iloc[-1])
df["dollar_vol"] = df["close"] * df["volume"]    # Filters out illiquid intervals
df["mean_trade_size"] = (
        df["volume"] / df["number_of_trades"].replace(0, np.nan)
    )
direction = np.sign(df["close"].diff()).fillna(0)          # -1, 0, +1
df["obv"] = (direction * df["volume"]).cumsum()
mfm = ((df["close"] - df["low"]) - (df["high"] - df["close"])) / (df["high"] - df["low"])  # money-flow
mfm = mfm.replace([np.inf, -np.inf], 0).fillna(0)          # handle hi==lo or NaN
df["ad_line"] = (mfm * df["volume"]).cumsum()

In [ ]:
# Time-based covariates
df["minute"] = df["open_time"].dt.minute
df["hour"] = df["open_time"].dt.hour
df["dayofweek"] = df["open_time"].dt.dayofweek
df["month"] = df["open_time"].dt.month
df["is_weekend"] = df["dayofweek"] >= 5
# cyclical encoding
df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)

In [ ]:
# Technical-indicator classics (non-linear combinations)
df["rsi_14"]  = ta.RSI(df["close"], timeperiod=14)
df["macd"], df["macd_signal"], df["macd_hist"] = ta.MACD(df["close"])
df["bb_upper"], df["bb_middle"], df["bb_lower"] = ta.BBANDS(df["close"])

In [ ]:
feature_cols = [c for c in df.columns if c != "log_ret_target"]
df_model = df.dropna(subset=["log_ret_target"])
X = df_model[feature_cols]
y = df_model["log_ret_target"]

print(X.columns)
non_numeric = X.select_dtypes(exclude=["number"]).columns
print("Non-numeric columns:", list(non_numeric))
X = X.drop(columns=non_numeric)
feature_cols = [c for c in X.columns]  # feature_cols update after non numeric cols drop
assert X.select_dtypes(exclude=["number"]).empty, "Still non-numeric cols!"
